# BART Summarization Notebook

This notebook compares pretrained BART and LoRA fine-tuned BART for abstractive news summarization on a 3,000-sample subset of CNN/DailyMail.

Expected setup:
- GPU runtime recommended
- Kaggle dataset access configured before running download cells
- Main metrics: ROUGE and BERTScore

In [ ]:
# Install necessary libraries
!pip install -q transformers datasets evaluate bert_score peft accelerate bitsandbytes kaggle

# Suppress warnings and import essentials
from IPython.display import display
from warnings import filterwarnings
filterwarnings('ignore')

import json
import re
import unicodedata
import os
import random
import pandas as pd
import torch

# Hugging Face Datasets and Evaluation
from datasets import Dataset
import evaluate

# Hugging Face Transformers & PEFT
from transformers import (
    BartTokenizer, BartForConditionalGeneration,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
# Upload your kaggle.json and enable Kaggle API
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and unzip dataset
!kaggle datasets download -d gowrishankarp/newspaper-text-summarization-cnn-dailymail --unzip

In [ ]:
# Load 3000 samples for consistency
df = pd.read_csv("cnn_dailymail/train.csv", nrows=3000)
df.columns = [str(c).strip() for c in df.columns]

# Preprocess text exactly as in Mistral
def clean_text(text):
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r'\s(\.)\s', ' ', text)
    text = re.sub(r"<[^>]+>", "", text)  # Remove HTML tags
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["article"] = df["article"].apply(clean_text)
df["highlights"] = df["highlights"].apply(clean_text)

# Convert to Hugging Face Dataset and split
dataset = Dataset.from_pandas(df)
dataset = dataset.shuffle(seed=42)
train_dataset = dataset.select(range(0, int(0.8 * len(dataset))))
test_dataset = dataset.select(range(int(0.8 * len(dataset)), len(dataset)))

In [ ]:
from transformers import pipeline

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn").to("cuda")

def generate_summary(example):
    inputs = tokenizer(example["article"], return_tensors="pt", max_length=1024, truncation=True).to("cuda")
    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=150,
        min_length=40,
        length_penalty=2.0,
        early_stopping=True
    )
    return {"generated_summary": tokenizer.decode(summary_ids[0], skip_special_tokens=True)}

test_dataset_pred = test_dataset.map(generate_summary)

In [ ]:
# Install missing dependencies for evaluation metrics
!pip install -q rouge_score
!pip install -q bert_score

# Load evaluation metrics
import evaluate
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# Prepare predictions and references
refs = test_dataset["highlights"]
preds = test_dataset_pred["generated_summary"]

# Compute ROUGE
rouge_output = rouge.compute(predictions=preds, references=refs)

# Compute BERTScore
bertscore_output = bertscore.compute(predictions=preds, references=refs, lang="en")

# Print results
print(" ROUGE Scores:")
for k, v in rouge_output.items():
    print(f"{k}: {v:.4f}")

print("\n BERTScore (F1):")
print(f"F1 Mean: {sum(bertscore_output['f1']) / len(bertscore_output['f1']):.4f}")

In [ ]:
def tokenize(example):
    model_inputs = tokenizer(example["article"], max_length=1024, truncation=True)
    labels = tokenizer(example["highlights"], max_length=150, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(tokenize, batched=True)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Apply LoRA on BART
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-lora",
    per_device_train_batch_size=2,
    eval_strategy="no",
    save_strategy="epoch",
    logging_dir="./logs",
    num_train_epochs=3,
    fp16=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
def generate_lora_summary(example):
    inputs = tokenizer(example["article"], return_tensors="pt", max_length=1024, truncation=True).to("cuda")

    # Use base_model from PEFT model
    summary_ids = peft_model.base_model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=150,
        min_length=40,
        length_penalty=2.0,
        early_stopping=True
    )

    return {"generated_summary_lora": tokenizer.decode(summary_ids[0], skip_special_tokens=True)}

# Run generation
test_dataset_pred_lora = test_dataset.map(generate_lora_summary)

# Evaluate
preds_lora = test_dataset_pred_lora["generated_summary_lora"]
refs_lora = test_dataset["highlights"]

rouge_output_lora = rouge.compute(predictions=preds_lora, references=refs_lora)
bertscore_output_lora = bertscore.compute(predictions=preds_lora, references=refs_lora, lang="en")

print(" LoRA BART ROUGE:")
for k, v in rouge_output_lora.items():

    print(f"{k}: {v:.4f}")

print("\n LoRA BART BERTScore (F1):")
print(f"F1 Mean: {sum(bertscore_output_lora['f1']) / len(bertscore_output_lora['f1']):.4f}")

In [ ]:
# Create a DataFrame for comparison
import pandas as pd

comparison_df = pd.DataFrame({
    "Article": test_dataset["article"],
    "Generated Summary": test_dataset_pred_lora["generated_summary_lora"],
    "Actual Summary": test_dataset["highlights"]
})

# Display first 5 rows
comparison_df.head()

In [ ]:
comparison_df.to_csv("bart_lora_predictions.csv", index=False)
from google.colab import files
files.download("bart_lora_predictions.csv")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Metric labels
metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "ROUGE-Lsum", "BERTScore (F1)"]

# Scores
mistral_scores = [0.1756, 0.0888, 0.1217, 0.1537, 0.8371]
bart_scores = [0.4036, 0.1959, 0.2937, 0.2937, 0.8781]

# X positions and bar width
x = np.arange(len(metrics))
bar_width = 0.35

# Plotting
plt.figure(figsize=(12, 6))
bars1 = plt.bar(x - bar_width/2, mistral_scores, width=bar_width, label='Mistral LoRA', color='skyblue')
bars2 = plt.bar(x + bar_width/2, bart_scores, width=bar_width, label='BART LoRA', color='lightcoral')

# Add numerical labels on top of bars
for bar in bars1:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.01, f'{height:.2f}', ha='center', va='bottom', fontsize=10)

for bar in bars2:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.01, f'{height:.2f}', ha='center', va='bottom', fontsize=10)

# Labels and layout
plt.xticks(x, metrics)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Comparison of Evaluation Metrics: BART LoRA vs Mistral LoRA")
plt.legend()
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Metrics (from your outputs)
metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "ROUGE-Lsum", "BERTScore (F1)"]
scores = [0.1756073149643942, 0.08875714258750812, 0.12168485959390071, 0.1536833051739854, 0.8370733261108398]

plt.figure(figsize=(10, 5))
plt.bar(metrics, scores)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Evaluation Metrics - Mistral LoRA")
plt.grid(axis='y')
plt.show()

In [ ]:
!pip install -q gradio

import gradio as gr

# Use the LoRA fine-tuned model
def summarize_with_bart(article):
    inputs = tokenizer(article, return_tensors="pt", max_length=1024, truncation=True).to("cuda")
    summary_ids = peft_model.base_model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=150,
        min_length=40,
        length_penalty=2.0,
        early_stopping=True
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

gr.Interface(
    fn=summarize_with_bart,
    inputs=gr.Textbox(lines=10, placeholder="Paste a news article here...", label="News Article"),
    outputs=gr.Textbox(label="Generated Summary"),
    title="BART News Summarizer (LoRA Fine-tuned)",
    description="This demo uses a LoRA fine-tuned BART model to generate summaries for news articles."
).launch()

In [ ]:
import gradio as gr

# Use the LoRA fine-tuned model
def summarize_with_bart(article):
    inputs = tokenizer(article, return_tensors="pt", max_length=1024, truncation=True).to("cuda")
    summary_ids = peft_model.base_model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=150,
        min_length=40,
        length_penalty=2.0,
        early_stopping=True
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

gr.Interface(
    fn=summarize_with_bart,
    inputs=gr.Textbox(lines=10, placeholder="Paste a news article here...", label="News Article"),
    outputs=gr.Textbox(label="Generated Summary"),
    title="Mistral News Summarizer (LoRA Fine-tuned)",
    description="This demo uses a LoRA fine-tuned Mistral model to generate summaries for news articles."
).launch()